# Titanic ⑤ モデルを選ぶ

04 で決めた特徴量（02 と同じ 7 列）と欠損の埋め方のまま，使うモデルを決めた記録．

**決定: 勾配ブースティング（scikit-learn の `HistGradientBoostingClassifier`）を使う．** ほかのモデルとは比べず，既定値でのスコアを確かめてから，06 でパラメータを調整する．

勾配ブースティングは，決定木を 1 本ずつ順に足していくモデル．新しい木は，それまでの木の予測で外れた分（損失の勾配）を小さくするように作られる．表形式のデータでは強いことが多い一方，パラメータが多く，データが少ないと過学習しやすい．

In [ ]:
# src/ 以下を編集したら，カーネルを再起動しなくても読み込み直す
%load_ext autoreload
%autoreload 2

import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import cross_validate

from titanic.data import load_data
from titanic.features import CATEGORICAL, IMPUTE_BY, NUMERIC, fill_embarked
from titanic.modeling import CV, build_pipeline

train, _ = load_data()
X, y = fill_embarked(train), train["Survived"]

## 既定値での結果

交差検証の正解率に加えて，train（学習に使った行）での正解率も見る．train での正解率が交差検証よりずっと高いときは，学習データを覚え込んでいる（過学習している）と考えられる．

In [ ]:
model = HistGradientBoostingClassifier()
result = cross_validate(build_pipeline(model, NUMERIC, CATEGORICAL, IMPUTE_BY), X, y, cv=CV, return_train_score=True, n_jobs=-1)

print(f"交差検証の正解率: 平均 {result['test_score'].mean():.3f} / 標準偏差 {result['test_score'].std():.3f}")
print(f"train の正解率  : 平均 {result['train_score'].mean():.3f}（交差検証との差 {result['train_score'].mean() - result['test_score'].mean():.3f}）")
print(f"1 回あたりの学習時間: {result['fit_time'].mean():.2f} 秒")

In [ ]:
# 過学習に関わる主なパラメータの既定値
params = model.get_params()
print(pd.Series({name: params[name] for name in ["learning_rate", "max_iter", "max_leaf_nodes", "max_depth", "min_samples_leaf", "l2_regularization", "early_stopping"]}))

交差検証の正解率は 0.820 で，04 の基準にしたランダムフォレスト（02 の設定，0.821）とほぼ同じだった．

一方，train での正解率は 0.947 で，交差検証と 0.127 も離れている．既定値は，葉の数が最大 31 の木を学習率 0.1 で 100 本足し，正則化（`l2_regularization`）はかけない設定で，891 行のデータには表現力が大きすぎるのだと思う．木を小さくする，学習率を下げる，正則化をかけるといった調整で過学習を抑える余地があるので，06 でパラメータを調整する．